In [1]:
"""
Today we'll build out another RAG application but we'll do a couple of key
differences. We'll plan to use a Document Loader, We'll do a few different
kinds of retrieval (Similarity, MMR, and Multi-Query Retrieval) and then
talk about structured inputs and outputs for our LLMs
"""

%pip install -qU "langchain>=1.0,<2.0" langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-chroma langchain-google-genai sentence-transformers chromadb pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.5/558.5 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60

In [2]:
"""
Here we'll make our files for today. Yesterday we used explicit python dictionaries
to define our documents. Today, we'll treat this like using an MD set of files for
our knowledge base. We still have to create them though because they don't exist.
"""

from pathlib import Path

support_folder = Path("it_support_knowledge_base")
support_folder.mkdir(exist_ok=True)

support_files = {
    "vpn_troubleshooting.md": """
# VPN Troubleshooting

A VPN status of Connected confirms that the tunnel was established. It does not
guarantee that every internal application is reachable or that the user's
single-sign-on session is valid.

For one employee who can browse public websites but cannot reach internal
applications:

1. Disconnect from the VPN.
2. Close the VPN client completely.
3. Remove any saved password from the VPN client.
4. Sign out of the corporate identity portal.
5. Restart the computer.
6. Connect again using the current password.
7. Test the corporate intranet before testing individual applications.

If no internal resource is reachable after these steps, collect the VPN client
logs and escalate to Network Operations.

If the intranet works but several single-sign-on applications fail, follow the
identity and access escalation process instead.
""",
    "password_reset_side_effects.md": """
# Password Reset Side Effects

After a corporate password reset, older credentials may remain cached in:

- The VPN client
- The operating-system credential manager
- Browser sessions
- Desktop applications
- Mobile email clients

A user should sign out of existing corporate sessions and update any stored
password before repeatedly retrying authentication.

Repeated use of the previous password may temporarily lock the account.

A password reset does not normally require a new MFA enrollment. Do not remove
the user's MFA methods unless the MFA troubleshooting guide specifically calls
for it.
""",
    "mfa_troubleshooting.md": """
# MFA Troubleshooting

MFA is functioning when the user receives a prompt and can approve it
successfully.

If no prompt appears, confirm that the user is signing in with the correct
corporate account and that the device time is accurate.

If the user receives repeated prompts that they did not initiate, stop normal
troubleshooting and escalate to the Security Operations Center.

Do not reset or remove MFA enrollment solely because a password changed.
""",
    "device_certificate_renewal.md": """
# Managed Device Certificate Renewal

Managed laptops use a device certificate for access to selected internal
applications.

Possible certificate symptoms include:

- VPN connects but selected internal applications reject the device
- The browser displays a device-trust or certificate message
- Access fails across several applications that require managed-device trust

First, confirm that the device is company managed. Then open the device
management client and run Sync. Restart the browser after synchronization.

If the certificate is expired or synchronization fails, escalate to Endpoint
Engineering. Do not manually install an unapproved certificate.
""",
    "known_service_incidents.md": """
# Known Service Incidents

Current classroom status:

- Corporate VPN: Operational
- Corporate identity provider: Operational
- GitHub Enterprise: Operational
- Payroll portal: Operational
- Corporate intranet: Operational

A payroll maintenance window is scheduled for 11:00 PM to 11:30 PM.

There is no active incident explaining a single employee's inability to access
GitHub Enterprise and payroll during normal business hours.
""",
    "application_access_matrix.md": """
# Application Access Matrix

GitHub Enterprise:
- Uses corporate single sign-on
- Requires an active employee account
- Requires membership in an approved engineering group
- Does not require the VPN when accessed from a managed device

Payroll portal:
- Uses corporate single sign-on
- Requires an active employee account
- Requires managed-device trust
- May be reached with or without the VPN from a managed device

Corporate intranet:
- Requires the VPN when offsite
- Uses corporate single sign-on

A Connected VPN status is not proof that the user is authorized for a specific
application.
""",
    "escalation_guide.md": """
# Access Issue Escalation Guide

Escalate to Network Operations when:
- The VPN cannot connect
- No internal network resource is reachable
- VPN logs show routing or tunnel failures

Escalate to Identity and Access Management when:
- Several single-sign-on applications fail for one user
- The issue began immediately after a password reset
- Account lockout or stale credential symptoms remain after sign-out and restart

Escalate to Endpoint Engineering when:
- A managed-device certificate is expired
- Device-management synchronization fails
- Several device-trust applications reject the laptop

Escalate to an application support team when:
- Only one application fails
- Other single-sign-on applications work
- The user may be missing an application-specific role or group
""",
    "ticket_priority_policy.md": """
# Ticket Priority Policy

P1 — Critical:
A company-wide outage, major security event, or critical production service is
unavailable to most users.

P2 — High:
A major business service is unavailable to a department or many users and no
reasonable workaround exists.

P3 — Standard:
One employee is blocked from multiple business applications, or one employee
cannot perform important work but the broader services remain operational.

P4 — Low:
A how-to question, minor inconvenience, information request, or issue with an
available workaround.

A single employee unable to access GitHub and payroll while the services remain
operational should normally begin as P3.
""",
}

for filename, content in support_files.items():
    path = support_folder / filename
    path.write_text(content.strip(), encoding="utf-8")

print("Created support files:")
for path in sorted(support_folder.glob("*.md")):
    print("-", path.name)

Created support files:
- application_access_matrix.md
- device_certificate_renewal.md
- escalation_guide.md
- known_service_incidents.md
- mfa_troubleshooting.md
- password_reset_side_effects.md
- ticket_priority_policy.md
- vpn_troubleshooting.md


In [5]:
"""
At this point we are at the "Starting" Step. Here we load the documents
with a DocumentLoader

DocumentLoader? Used to load documents of all types into a standarized format
known as a LangChain Document.
Contains: Metadata and Page Content

In our case we're just reading in Text documents so we'll leverage the TextLoader.
But there are specific loaders for other types or sources of files including
Google Drive and S3 Buckets (Simple Storage Service from AWS). There are also
file specific loaders like PDfLoader or CSVLoader as well.
"""

from langchain_community.document_loaders import TextLoader

loaded_documents = []

for file_path in sorted(support_folder.glob("*.md")):
  # To load the documents is pretty simple
  loader = TextLoader(
      str(file_path),
      encoding="utf-8"
  )
  loaded_documents.extend(loader.load())

print("Loaded Documents: ", len(loaded_documents))

Loaded Documents:  8


In [6]:
"""
Let's inspect one of these documents to see what it contains
"""
first_document = loaded_documents[0]

print(first_document.page_content)
print(first_document.metadata)
print(type(first_document))

# Application Access Matrix

GitHub Enterprise:
- Uses corporate single sign-on
- Requires an active employee account
- Requires membership in an approved engineering group
- Does not require the VPN when accessed from a managed device

Payroll portal:
- Uses corporate single sign-on
- Requires an active employee account
- Requires managed-device trust
- May be reached with or without the VPN from a managed device

Corporate intranet:
- Requires the VPN when offsite
- Uses corporate single sign-on

A Connected VPN status is not proof that the user is authorized for a specific
application.
{'source': 'it_support_knowledge_base/application_access_matrix.md'}
<class 'langchain_core.documents.base.Document'>


In [7]:
"""
At this point our documents are LangChain Document objects. It doesn't matter now
if the origin was a text file or a google drive pull or an S3 Bucket or if it
was a PDF or text or a CSV, Langchain can now interact with all of them in the
same way.

Technically at this point we could leverage the documents as-is and they would
work, but I think the metadata is a little lacking and we could clean them
to make it a little bit more normalized (consistent).

These transformations to the loaded data are known Data Transformations. We
will see proper Data Transformers later, but for now this is a manual process
"""

# I'm going to add some manual metadata to our documents for better filtering
# and things later

metadata_by_source = {
    "vpn_troubleshooting.md": {
        "product": "vpn",
        "document_type": "runbook",
        "audience": "employee",
    },
    "password_reset_side_effects.md": {
        "product": "identity",
        "document_type": "knowledge_article",
        "audience": "employee",
    },
    "mfa_troubleshooting.md": {
        "product": "identity",
        "document_type": "runbook",
        "audience": "service_desk",
    },
    "device_certificate_renewal.md": {
        "product": "endpoint",
        "document_type": "runbook",
        "audience": "service_desk",
    },
    "known_service_incidents.md": {
        "product": "operations",
        "document_type": "status",
        "audience": "service_desk",
    },
    "application_access_matrix.md": {
        "product": "applications",
        "document_type": "reference",
        "audience": "service_desk",
    },
    "escalation_guide.md": {
        "product": "support",
        "document_type": "policy",
        "audience": "service_desk",
    },
    "ticket_priority_policy.md": {
        "product": "support",
        "document_type": "policy",
        "audience": "service_desk",
    },
}

In [8]:
import re
from langchain_core.documents import Document

transformed_documents = []

# We'll iterate over the existing loaded documents and clean them by
# normalizing the whitespace and add in the metadata

for document in loaded_documents:
  source_name = Path(document.metadata["source"]).name

  # Clean the text by normalizing the whitespace
  # If there are 3 or more new lines in a row, we'll drop it down to 2
  cleaned_text = re.sub(r"\n{3,}", "\n\n", document.page_content).strip()

  # Add the document with the appropriate metadaa to our list
  transformed_documents.append(
      Document(
          page_content = cleaned_text,
          metadata = {
              "source": source_name,
              **metadata_by_source[source_name]
          }
      )
  )

# At this point the transformation should be complete
print(transformed_documents[0].page_content)
print(transformed_documents[0].metadata)

# Application Access Matrix

GitHub Enterprise:
- Uses corporate single sign-on
- Requires an active employee account
- Requires membership in an approved engineering group
- Does not require the VPN when accessed from a managed device

Payroll portal:
- Uses corporate single sign-on
- Requires an active employee account
- Requires managed-device trust
- May be reached with or without the VPN from a managed device

Corporate intranet:
- Requires the VPN when offsite
- Uses corporate single sign-on

A Connected VPN status is not proof that the user is authorized for a specific
application.
{'source': 'application_access_matrix.md', 'product': 'applications', 'document_type': 'reference', 'audience': 'service_desk'}


In [10]:
"""
Let's Chunk the data to make it easier for our embedding store to create specific
vectors for them
"""

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=420,
    chunk_overlap=70
)

chunks = splitter.split_documents(transformed_documents)

In [11]:
"""
Next step is basically the exact same as yesterday. As a reminder we can swap
out our implementations for Vector DB with minimal changes later down the line.
This flexibility is the power of LangChain
"""

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

collection_name = "it_support_documents"

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=collection_name
)

print("Stored Chunks: ", len(chunks))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stored Chunks:  18


In [17]:
support_question = (
    "I reset my password this morning. The VPN says it is connected, "
    "but I cannot access GitHub or the Payroll Portal. Other websites work. "
    "What should I try, and when should this be escalated?"
)

# We can find some relevant results by querying based off similarity
baselin_results = vector_store.similarity_search(
    support_question,
    k=5,
    # We can add in metadata filtering if we wanted
    # filter={"product": "vpn"}
)

for index, document in enumerate(baselin_results, start=1):
  print("Result: ", index)
  print(document.metadata)
  print(document.page_content)
  print("-"*100)

Result:  1
{'product': 'vpn', 'audience': 'employee', 'source': 'vpn_troubleshooting.md', 'document_type': 'runbook'}
1. Disconnect from the VPN.
2. Close the VPN client completely.
3. Remove any saved password from the VPN client.
4. Sign out of the corporate identity portal.
5. Restart the computer.
6. Connect again using the current password.
7. Test the corporate intranet before testing individual applications.
----------------------------------------------------------------------------------------------------
Result:  2
{'document_type': 'runbook', 'product': 'vpn', 'source': 'vpn_troubleshooting.md', 'audience': 'employee'}
If no internal resource is reachable after these steps, collect the VPN client
logs and escalate to Network Operations.

If the intranet works but several single-sign-on applications fail, follow the
identity and access escalation process instead.
----------------------------------------------------------------------------------------------------
Result:  3
{'

In [18]:
"""
At this point we've been able to do similarity searching on our documents and
their embedded vectors. Now we're going to use a process known as MMR for
searching in our retrievers.

What is MMR?
Maximal-Marginal Relevance. It means it will get the documents that are
relevant and then re-rank them based off the initial similarity and penalize them
for being too similar to other documents.

This helps prevent the LLM from receiving 3 almost identical object and helps
ensure you've reached better coverage for your docs.
"""

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs = {"k": 5}
)

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs = {
        "k": 5,
        "fetch_k" : 15,
        # Lambda = 1 -> No MMR
        # Lambda = 0 -> Maximum diversity
        # .5 is a good starting point
        "lambda_mult" : .5
    }
)

print("Similarity Sources")
for document in similarity_retriever.invoke(support_question):
  print("-", document.metadata["source"])

print("MMR Sources")
for document in mmr_retriever.invoke(support_question):
  print("-", document.metadata["source"])

Similarity Sources
- vpn_troubleshooting.md
- vpn_troubleshooting.md
- known_service_incidents.md
- escalation_guide.md
- password_reset_side_effects.md
MMR Sources
- vpn_troubleshooting.md
- known_service_incidents.md
- escalation_guide.md
- application_access_matrix.md
- mfa_troubleshooting.md


In [19]:
"""
Here we've used MMR to gather more relevant information before it gets passed to
our LLM. Another technique we can use is Multi-Query Retrieval.
This means leverage multiple queries that are similarly worded to cover the scope
of the issue and get the relevant details.

We're actaully going to leverage an LLM to create the different phrases
"""

import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI


os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite",
    temperature = 0.0,
    max_tokens= 800,
    max_retries = 2
)

In [20]:
"""
At this point we have looped in our LLM. Well we want our LLM to generate similar
queries to the question that we initially asked to help us get better results
from our vector db. So we need to call our llm for this and we want the results
to be structured

We'll explicitly define the structure using Pydantic
Pydantic is a library for data validation and parsing, you extend a BaseModel
and define essentially what fields you want and what they look like
"""

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# We'll create a simple class to define the structure of our LLM response
class SearchQueries(BaseModel):
  # In here we'll define our fields
  queries: list[str] = Field(
      description="Exactly 4 distinct knowledge-base search queries"
  )

# Create our prompt for our LLM to generate different queries
query_prompt = ChatPromptTemplate.from_template("""
Create exactly four short search queries for an IT support knowledge base.

The queries should cover different information needs contained in the user's
ticket. Do not answer the ticket.

USER TICKET:
{question}
""")

# Define our query model with our return schema. Yesterday we used a regular
# standard llm output, today we're controlling it
query_model = llm.with_structured_output(
    schema=SearchQueries.model_json_schema(),
    method="json_schema"
)

# Create a simple chain to pass the question to our model
query_generation_chain = query_prompt | query_model

In [21]:
generated_queries = query_generation_chain.invoke({
    "question": support_question
})

print("Original Ticket")
print(support_question)

print("Generated Search Queries")
for query in generated_queries["queries"]:
  print("-",query)

Original Ticket
I reset my password this morning. The VPN says it is connected, but I cannot access GitHub or the Payroll Portal. Other websites work. What should I try, and when should this be escalated?
Generated Search Queries
- VPN connected but cannot access internal resources
- troubleshoot GitHub access issues after password reset
- payroll portal connection error after VPN login
- IT support ticket escalation criteria for network access


In [23]:
"""
At this point we need to apply all of the different steps so I'm gonna create a function
so we can leverage this later which will take in our initial question, create the
alternate queries, and then get the relevant data from the vector db

This is called Multi-Query Retrieval
"""

def retrieve_with_multiple_queries(question):
  generated = query_generation_chain.invoke({
      "question": question
  })

  # Group all queries together
  queries = [
      question,
      *generated["queries"]
  ]

  # We need to search up each value and get the documents without duplicate
  documents = []
  seen = set()

  for query in queries:
    for document in mmr_retriever.invoke(query):
      key = (
          document.metadata["source"],
          document.page_content
      )

      if key not in seen:
        seen.add(key)
        documents.append(document)

  return documents

In [24]:
retrieve_with_multiple_queries(support_question)

[Document(id='024fabea-a2bc-4914-9caa-48e43b99e745', metadata={'document_type': 'runbook', 'audience': 'employee', 'product': 'vpn', 'source': 'vpn_troubleshooting.md'}, page_content='1. Disconnect from the VPN.\n2. Close the VPN client completely.\n3. Remove any saved password from the VPN client.\n4. Sign out of the corporate identity portal.\n5. Restart the computer.\n6. Connect again using the current password.\n7. Test the corporate intranet before testing individual applications.'),
 Document(id='b717cf3c-369f-4e1a-9719-caf416e6d90b', metadata={'audience': 'service_desk', 'product': 'operations', 'document_type': 'status', 'source': 'known_service_incidents.md'}, page_content="A payroll maintenance window is scheduled for 11:00 PM to 11:30 PM.\n\nThere is no active incident explaining a single employee's inability to access\nGitHub Enterprise and payroll during normal business hours."),
 Document(id='47b1282f-6037-4ff5-be01-694bce671b16', metadata={'product': 'support', 'audience

In [29]:
"""
Tradeoffs to using Multi-Query Retrieval
- Additional Model call so if you're strapped for resource limits, be careful
- More vector searches
- More latency
- More opportunities to retreive irrelevant data

At this point we have gotten ready all of the data that we need (or have a function
to do exactly that) so we need to take the results and organize them so we can hand it
to our LLM model
"""

def format_documents(documents):
  return "\n\n".join(
      f"Source: {document.metadata['source']}\n"
      f"Product: {document.metadata["product"]}\n"
      f"Document Type: {document.metadata["document_type"]}\n"
      f"Content: {document.page_content}"
      for document in documents
  )

In [30]:
context = format_documents(retrieve_with_multiple_queries(support_question))

print(context)

Source: vpn_troubleshooting.md
Product: vpn
Document Type: runbook
Content: 1. Disconnect from the VPN.
2. Close the VPN client completely.
3. Remove any saved password from the VPN client.
4. Sign out of the corporate identity portal.
5. Restart the computer.
6. Connect again using the current password.
7. Test the corporate intranet before testing individual applications.

Source: known_service_incidents.md
Product: operations
Document Type: status
Content: A payroll maintenance window is scheduled for 11:00 PM to 11:30 PM.

There is no active incident explaining a single employee's inability to access
GitHub Enterprise and payroll during normal business hours.

Source: escalation_guide.md
Product: support
Document Type: policy
Content: # Access Issue Escalation Guide

Escalate to Network Operations when:
- The VPN cannot connect
- No internal network resource is reachable
- VPN logs show routing or tunnel failures

Source: application_access_matrix.md
Product: applications
Document 

In [32]:
"""
Now that we have all of the relevant documents and they're formatted correctly
we need to pass the information to an LLM prompt and get the response

We'll briefly talk about prompt serialization
This is the process of saving a prompt in a way that allows it to be shared and
reused in a defined format. TLDR, we're saving prompts as JSON files to be shared
if needed.
"""

import json

answer_template = """
You are an IT support assistant using an internal knowledge base.

Use only the supplied context.

Requirements:

1. Do not claim that a Connected VPN proves application access is working.
2. Do not invent an active outage.
3. Do not recommend removing MFA solely because a password changed.
4. Recommend only steps supported by the context.
5. Select escalation and priority rules from the supplied policies.
6. State what additional information is still needed.
7. If the context does not establish an answer, say that it is unknown.

CONTEXT:
{context}

USER TICKET:
{question}
"""

prompt_config = {
    "name":"it-support-resolution",
    "version":"1.0",
    "input_variables": ["context", "question"],
    "template": answer_template.strip()
}

# Serialize our prompt by just writing it to a file that can be loaded wherever
prompt_path = Path("it_support_prompt_v1.json")
prompt_path.write_text(
    json.dumps(prompt_config, indent=2),
    encoding="utf-8"
)

731

In [33]:
# Imagine we already had the prompt ready to go and we just needed to load it
loaded_prompt_config = json.loads(
    prompt_path.read_text(encoding="utf-8")
)

answer_prompt = ChatPromptTemplate.from_template(
    loaded_prompt_config["template"]
)

In [35]:
"""
Again, yesterday we used a regular string output parser but we can define the exact
schema we want our answer in by giving it a class we created with Pydantic

Why is this important? Having a structured response will allow us to create
a frontend for this piece and guarantee the correct infomation is being passed
"""

from typing import Literal

class SupportResolution(BaseModel):
  summary:str
  probable_causes: list[str]
  recommend_steps: list[str]
  escalation_team: str | None
  ticket_priority: Literal["P1", "P2", "P3", "P4"]
  missing_information: list[str]
  need_human_review: bool

structured_answer_model = llm.with_structured_output(
    schema=SupportResolution.model_json_schema(),
    method="json_schema"
)

answer_chain = (
    answer_prompt | structured_answer_model
)

In [40]:
"""
At this point we have all of the pieces to complete our RAG application
Let's put them together and then create a function to test it
"""

documents = retrieve_with_multiple_queries(support_question)

context = format_documents(documents)

answer = answer_chain.invoke({
    "context": context,
    "question": support_question
})

context_sources = sorted({
    document.metadata["source"]
    for document in documents
})

result = {
    "question": support_question,
    "resolution": answer,
    "context_sources" : context_sources
}

result

{'question': 'I reset my password this morning. The VPN says it is connected, but I cannot access GitHub or the Payroll Portal. Other websites work. What should I try, and when should this be escalated?',
 'resolution': {'summary': 'User is unable to access GitHub Enterprise and the Payroll Portal following a recent password reset, despite the VPN showing as connected and public websites being accessible.',
  'probable_causes': ['Cached credentials in the VPN client, browser, or operating system',
   'Stale single-sign-on session',
   'Managed-device certificate issues or synchronization failure',
   'Account lockout due to repeated use of previous password'],
  'recommend_steps': ['Disconnect from the VPN and close the VPN client completely',
   'Remove any saved passwords from the VPN client',
   'Sign out of the corporate identity portal',
   'Restart the computer',
   'Connect to the VPN using the current password',
   'Test the corporate intranet',
   'Confirm the device is compan

In [41]:
def ask_support_knowledge(question):
  documents = retrieve_with_multiple_queries(question)
  context = format_documents(documents)
  resolution = answer_chain.invoke({
      "context": context,
      "question": question
  })

  return resolution

In [43]:
support_result = ask_support_knowledge(
    "I changed my password today. My VPN connects but GitHub and Payroll both reject me. Public websites work, what should I do?"
)

support_result

{'summary': 'The user is unable to access GitHub Enterprise and the payroll portal following a recent password change, despite the VPN connecting successfully and public websites remaining accessible.',
 'probable_causes': ['Cached credentials in the VPN client, browser, or operating system',
  'Expired or invalid device certificate affecting managed-device trust',
  'Stale browser sessions or application-specific authentication tokens',
  'Account lockout due to repeated use of previous password'],
 'recommend_steps': ['Disconnect from the VPN and close the VPN client completely',
  'Remove any saved passwords from the VPN client',
  'Sign out of the corporate identity portal and all active browser sessions',
  'Restart the computer',
  'Reconnect to the VPN using the current password',
  'Test access to the corporate intranet before testing individual applications'],
 'escalation_team': 'Identity and Access Team',
 'ticket_priority': 'P3',
 'missing_information': ['Confirmation of wh